# SSI FastConnect — kiểm tra REST + luồng tool của TradingAgents

Notebook này kiểm tra theo thứ tự: (1) kết nối SSI chỉ đọc, (2) lựa chọn nguồn dữ liệu trong router, (3) tool giá OHLCV và tool RSI mà các agent thực sự gọi. Không gọi LLM, không đặt lệnh.

**Credentials:** đọc từ biến môi trường runtime trước, rồi Colab Secrets; chỉ hỏi nhập ẩn nếu chưa có. Chúng chỉ ở bộ nhớ runtime, không được in ra hoặc lưu vào notebook/GitHub. Muốn xóa khỏi phiên, chọn Runtime → Disconnect and delete runtime.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/PhamHuuQan/TradingAgents.git"
BRANCH = "feature/vietnam-market-foundation"
REPO_DIR = Path("/content/TradingAgents")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)

# Install the project dependencies so this checks the actual agent tools/router,
# not only the standalone SSI connector.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], cwd=str(REPO_DIR), check=True)
sys.path.insert(0, str(REPO_DIR))
print("TradingAgents branch checked out and project dependencies installed.")

In [ ]:
from datetime import datetime, timedelta
from getpass import getpass
from zoneinfo import ZoneInfo
import os

def read_colab_secret(name):
    # Reuse the current runtime first, then Colab Secrets.
    value = os.environ.get(name)
    if value and value.strip():
        return value.strip()
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value.strip() if isinstance(value, str) and value.strip() else None

api_key = read_colab_secret("SSI_API_KEY") or getpass("SSI_API_KEY (input hidden): ")
api_secret = read_colab_secret("SSI_API_SECRET") or getpass("SSI_API_SECRET (input hidden): ")
if not api_key.strip() or not api_secret.strip():
    raise ValueError("Both SSI credentials are required; no credentials were saved to the notebook.")

os.environ["SSI_API_KEY"] = api_key.strip()
os.environ["SSI_API_SECRET"] = api_secret.strip()
del api_key, api_secret

today_vn = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()
from_date = (today_vn - timedelta(days=30)).isoformat()
to_date = today_vn.isoformat()

print(f"[1/3] Direct SSI REST smoke test: FPT, {from_date} to {to_date}.")
from tradingagents.dataflows.vietnam_market.smoke import main as smoke_main
exit_code = smoke_main(["--symbol", "FPT", "--from-date", from_date, "--to-date", to_date, "--page-size", "100"])
if exit_code != 0:
    raise RuntimeError("SSI REST smoke test failed. Read the sanitized error above; do not share credentials.")

print("\n[2/3] Verify that TradingAgents selected SSI for both prices and technical indicators.")
from tradingagents.dataflows.config import set_config
from tradingagents.dataflows.router import get_vendor
# Explicitly set the run's data source even if modules were imported earlier.
set_config({"data_vendors": {"core_stock_apis": "ssi", "technical_indicators": "ssi"}})
selected_prices = get_vendor("core_stock_apis", "get_stock_data")
selected_indicators = get_vendor("technical_indicators", "get_indicators")
print(f"Price-data vendor: {selected_prices}")
print(f"Indicator vendor: {selected_indicators}")
if selected_prices.split(",")[0].strip() != "ssi" or selected_indicators.split(",")[0].strip() != "ssi":
    raise RuntimeError("TradingAgents did not select SSI as the primary vendor; refusing to claim integration success.")

print("\n[3/3] Call the actual agent tools (no LLM/API bill; read-only market data only).")
from tradingagents.agents.tools import get_stock_data, get_indicators
price_result = get_stock_data.func("FPT", from_date, to_date, to_date)
if not isinstance(price_result, str) or "# SSI FastConnect V3 read-only OHLCV for FPT" not in price_result:
    raise RuntimeError("Agent get_stock_data tool did not return an SSI-sourced response.")
price_rows = [
    line for line in price_result.splitlines()
    if line.strip() and not line.startswith("#") and not line.startswith("Date,")
]
if not price_rows:
    raise RuntimeError("Agent get_stock_data returned no OHLCV rows.")
print(f"PASS — agent get_stock_data uses SSI and returned {len(price_rows)} OHLCV rows.")

indicator_result = get_indicators.func("FPT", "rsi", to_date, 30, to_date)
if not isinstance(indicator_result, str) or "# SSI FastConnect V3 indicator rsi for FPT" not in indicator_result:
    raise RuntimeError("Agent get_indicators tool did not return an SSI-sourced indicator.")
indicator_rows = [
    line for line in indicator_result.splitlines()
    if line.strip() and not line.startswith("#") and not line.startswith("Date,")
]
if not indicator_rows:
    raise RuntimeError("Agent get_indicators returned no RSI values.")
print(f"PASS — agent get_indicators uses SSI and returned {len(indicator_rows)} RSI points.")
print("\n✅ COMPLETE: SSI REST, data routing, and both agent tools passed read-only checks.")

# Credentials remain only in this Colab runtime's memory; they are never printed or saved to disk.
# Disconnect/delete the runtime to clear them.


## Bước tiếp theo: chạy một lượt phân tích AI thật

Ô kế tiếp ưu tiên thử Gemini 3.6 Flash trước vì lần chạy trước Gemini 3.8 Flash trả `503 UNAVAILABLE`. Nếu model quá tải, notebook thử lại một lần rồi tự chuyển sang Flash-Lite và model dự phòng khác. Key `AQ.` là loại Authorization API Key mới của Google; chương trình không in hoặc ghi key vào notebook.

Notebook dùng model ID trong [danh mục chính thức Gemini API](https://ai.google.dev/gemini-api/docs/models). Lượt chạy AI vẫn có thể gặp giới hạn Free Tier; nếu tất cả các model bị từ chối thì notebook sẽ dừng trước khi chạy graph.

In [ ]:
from datetime import datetime
from copy import deepcopy
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from zoneinfo import ZoneInfo
from getpass import getpass
import os

# Prefer the newest model, but automatically fall back if it is overloaded/unavailable.
# These model IDs appear in Google's current Gemini API model catalog.
MODEL_CANDIDATES = [
    # 3.8 returned 503 UNAVAILABLE in the last run, so try another stable Flash
    # model first; only come back to it after the lower-load alternatives.
    "gemini-3.6-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-3.8-flash",
]

# Read GOOGLE_API_KEY from the runtime or Colab Secrets; prompt hidden only if absent.
google_key = read_colab_secret("GOOGLE_API_KEY") or getpass("GOOGLE_API_KEY (input hidden): ")
if not google_key.strip():
    raise ValueError("GOOGLE_API_KEY is empty.")
os.environ["GOOGLE_API_KEY"] = google_key.strip()
del google_key

# Validate the key and select a working model BEFORE constructing the multi-agent graph.
# 503/429/5xx are transient: retry with backoff, then move to the next model.
# Authentication/permission failures are not retried as model-capacity issues.
import time
from langchain_google_genai import ChatGoogleGenerativeAI

def _safe_gemini_error(exc):
    text = str(exc)
    key = os.environ.get("GOOGLE_API_KEY", "")
    if key:
        text = text.replace(key, "[REDACTED]")
    return text[:500]

def _is_transient_gemini_error(exc):
    text = (type(exc).__name__ + " " + str(exc)).upper()
    return any(token in text for token in (
        "503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED",
        "500", "502", "504", "INTERNAL", "DEADLINE_EXCEEDED",
        "SERVICE UNAVAILABLE", "TIMED OUT",
    ))

MODEL_ID = None
model_errors = []
for candidate in MODEL_CANDIDATES:
    model_ok = False
    for attempt in range(2):  # at most one retry per model
        try:
            ping = ChatGoogleGenerativeAI(
                model=candidate,
                google_api_key=os.environ["GOOGLE_API_KEY"],
                max_retries=0,
                max_output_tokens=24,
                timeout=45,
            ).invoke("Reply with exactly the single word READY.")
            if not getattr(ping, "content", None):
                raise RuntimeError("Gemini returned an empty response.")
            MODEL_ID = candidate
            model_ok = True
            print(f"✅ Gemini API preflight passed: {MODEL_ID}")
            break
        except Exception as exc:
            safe_error = _safe_gemini_error(exc)
            model_errors.append((candidate, type(exc).__name__, safe_error))
            upper = (type(exc).__name__ + " " + str(exc)).upper()
            if any(token in upper for token in (
                "401", "403", "API_KEY_INVALID", "PERMISSION_DENIED",
                "API KEY NOT VALID", "BILLING",
            )):
                raise RuntimeError(
                    f"Gemini authentication/project-access failed for {candidate} "
                    f"({type(exc).__name__}): {safe_error}"
                ) from None
            if attempt == 0 and _is_transient_gemini_error(exc):
                delay = 2
                print(f"⚠️ {candidate} is temporarily unavailable; retrying once in {delay}s.")
                time.sleep(delay)
                continue
            print(f"⚠️ {candidate} unavailable ({type(exc).__name__}); trying the next supported candidate.")
            break
    if model_ok:
        break

if MODEL_ID is None:
    summary = "; ".join(
        f"{name}: {kind} — {message}" for name, kind, message in model_errors[-8:]
    )
    raise RuntimeError(
        "Gemini preflight failed for every configured model. "
        "No TradingAgents run was started. Last sanitized errors: " + summary[:1800]
    )

from tradingagents.default_config import DEFAULT_CONFIG
from tradingagents.graph.trading_graph import TradingAgentsGraph

# Explicit run config: Gemini for both tiers, SSI for Vietnamese equity price/indicators,
# one market analyst for the first end-to-end test, and bounded debate/tool rounds.
config = deepcopy(DEFAULT_CONFIG)
config["llm_provider"] = "google"
config["quick_think_provider"] = None
config["deep_think_provider"] = None
config["quick_think_llm"] = MODEL_ID
config["deep_think_llm"] = MODEL_ID
config["output_language"] = "Vietnamese"
config["max_debate_rounds"] = 1
config["max_risk_discuss_rounds"] = 1
config["max_tool_rounds"] = 4
config["max_recur_limit"] = 100
config["max_tokens"] = 2048
config["llm_max_retries"] = 1
config["data_vendors"] = dict(config.get("data_vendors", {}))
config["data_vendors"]["core_stock_apis"] = "ssi"
config["data_vendors"]["technical_indicators"] = "ssi"

# This adapter blocks backdated SSI data because historical first-availability
# timestamps are not known. Use the current Vietnam market date for this live test.
analysis_date = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date().isoformat()
print(f"Starting TradingAgents end-to-end test: FPT, {analysis_date}")
print("Selected analyst: market. Graph still runs research, trader, risk and portfolio stages.")
print("Data sources: SSI prices/indicators; other categories retain the repository config.")

ta = TradingAgentsGraph(
    selected_analysts=["market"],
    config=config,
    debug=False,
)
final_state, signal = ta.propagate("FPT", analysis_date, asset_type="stock")
report_path = Path(ta.save_reports(final_state, "FPT", html=True))
report_dir = report_path.parent if report_path.is_file() else report_path

decision = final_state.get("final_trade_decision", "")
print("\\n✅ TradingAgents graph completed.")
print(f"Signal/rating: {signal}")
print(f"Final trade decision: {str(decision)[:2500]}")
print(f"Report file: {report_path}")
print(f"Report directory: {report_dir}")


In [ ]:
# Package an EXISTING report only. This cell does not call Gemini or SSI.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

# Reuse the report from this runtime, including the earlier notebook variable
# where report_dir was mistakenly assigned the complete_report.md file itself.
candidate = globals().get("report_path")
if candidate is None:
    candidate = globals().get("report_dir")

if candidate is not None:
    candidate = Path(candidate)
else:
    candidate = Path("/root/.tradingagents/logs/reports")

if candidate.is_file():
    report_file = candidate
    report_dir = candidate.parent
elif candidate.is_dir() and candidate.name != "reports":
    report_dir = candidate
    report_file = report_dir / "complete_report.md"
else:
    reports_root = Path("/root/.tradingagents/logs/reports")
    recent_reports = sorted(
        reports_root.glob("FPT_*/complete_report.md"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not recent_reports:
        raise FileNotFoundError(
            "No completed FPT report found in this Colab runtime. "
            "Do not rerun the AI yet; the runtime may have been reset."
        )
    report_file = recent_reports[0]
    report_dir = report_file.parent

if not report_file.is_file():
    raise FileNotFoundError(f"Expected report file not found: {report_file}")

archive = Path("/content/TradingAgents_FPT_report.zip")
items = sorted(p for p in report_dir.rglob("*") if p.is_file())
if report_file not in items:
    items.append(report_file)
items = sorted(set(items))
if not items:
    raise RuntimeError(f"No files found under report directory: {report_dir}")

with ZipFile(archive, "w", compression=ZIP_DEFLATED) as zf:
    for item in items:
        zf.write(item, arcname=f"{report_dir.name}/{item.relative_to(report_dir)}")

# Validate the archive before offering it for download.
with ZipFile(archive, "r") as zf:
    names = zf.namelist()
    corrupt = zf.testzip()
    if not names or corrupt:
        raise RuntimeError(f"ZIP validation failed; empty={not names}, corrupt_member={corrupt}")

print(f"✅ ZIP validated: {len(names)} files, {archive.stat().st_size:,} bytes")
print("Files included:")
for name in names:
    print(" -", name)
print("Starting browser download...")
files.download(str(archive))
